# Section VIII Open Challenges and Research Roadmap Evidence Lab (Full-Scan + JSON/Markdown Fusion)


In [1]:
# @title 1. Install Dependencies
!pip install -q groq rapidfuzz tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 29.9 MB/s eta 0:00:00


In [2]:
# @title 2. Setup & Mount Drive
from google.colab import drive, userdata
import os, re, json, glob
from pathlib import Path
import pandas as pd
from tqdm import tqdm
from rapidfuzz import fuzz

drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/AKU_WorkSpace/survey_fdgit/OISAC_PRISMA_COMST'
if os.path.exists(BASE_DIR):
    os.chdir(BASE_DIR)
    print('Working dir:', os.getcwd())
else:
    print('Path not found:', BASE_DIR)


Mounted at /content/drive
Working dir: /content/drive/MyDrive/AKU_WorkSpace/survey_fdgit/OISAC_PRISMA_COMST


In [3]:
# @title 3. Config
PROCESSED_MD_DIR = Path('data/proc_markdowns')
JSON_DIR = Path('data/ext_res_v4')
UNIFIED_CSV = Path('data/ext_v4_uni.csv')
OUTPUT_DIR = Path('analysis/VIII_ev_v1')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_PROFILE = 'FULL_RESCAN'
NORMALIZE_LABELS = True
TARGET_PAPERS = None
LIMIT = None
LLM_CALLS = True

MODEL_PASS1 = 'meta-llama/llama-4-scout-17b-16e-instruct'
MODEL_PASS2 = 'llama-3.3-70b-versatile'
MODEL_VARIANT_GEN = MODEL_PASS1
USE_ESCALATION = True
ESCALATE_LABELS = {'INDIRECT', 'NONE', 'WEAK'}

RPM_BY_MODEL = {MODEL_PASS1: 120, MODEL_PASS2: 40, MODEL_VARIANT_GEN: 120}
DEFAULT_RPM = 30
MAX_RETRIES = 5
RETRY_BASE_SECONDS = 2.0

MAX_VARIANTS_PER_CONCEPT = 14
MAX_HITS_PER_CONCEPT_PER_PAPER = 6
MAX_CONTEXT_CHARS = 1200
CLASSIFY_CHUNK_SIZE = 4
BATCH_SIZE_PAPERS = 10

RESUME = True
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

SCHEMA_MAP_PATH = Path('analysis/II_sch_map.md')
GOV_PATH = Path('analysis/II_met_gov.md')
II_TRADEOFF_GOV_PATH = Path('analysis/II_trade_gov_2E.md')
II_METRIC_EVAL_GOV_PATH = Path('analysis/II_met_eval_gov_2D.md')

V_AXIS_PATH = Path('analysis/V_ev_v2/axis_definitions.md')
V_MAP_PATH = Path('analysis/V_ev_v2/mapping_rules.md')
VI_AXIS_PATH = Path('analysis/VI_ev_v2/axis_definitions.md')
VI_MAP_PATH = Path('analysis/VI_ev_v2/mapping_rules.md')
VII_AXIS_PATH = Path('analysis/VII_ev_v2/axis_definitions.md')
VII_MAP_PATH = Path('analysis/VII_ev_v2/mapping_rules.md')

schema_text = SCHEMA_MAP_PATH.read_text(encoding='utf-8', errors='ignore') if SCHEMA_MAP_PATH.exists() else ''
gov_text = GOV_PATH.read_text(encoding='utf-8', errors='ignore') if GOV_PATH.exists() else ''
ii_tradeoff_text = II_TRADEOFF_GOV_PATH.read_text(encoding='utf-8', errors='ignore') if II_TRADEOFF_GOV_PATH.exists() else ''
ii_eval_text = II_METRIC_EVAL_GOV_PATH.read_text(encoding='utf-8', errors='ignore') if II_METRIC_EVAL_GOV_PATH.exists() else ''
v_axis_text = V_AXIS_PATH.read_text(encoding='utf-8', errors='ignore') if V_AXIS_PATH.exists() else ''
v_map_text = V_MAP_PATH.read_text(encoding='utf-8', errors='ignore') if V_MAP_PATH.exists() else ''
vi_axis_text = VI_AXIS_PATH.read_text(encoding='utf-8', errors='ignore') if VI_AXIS_PATH.exists() else ''
vi_map_text = VI_MAP_PATH.read_text(encoding='utf-8', errors='ignore') if VI_MAP_PATH.exists() else ''
vii_axis_text = VII_AXIS_PATH.read_text(encoding='utf-8', errors='ignore') if VII_AXIS_PATH.exists() else ''
vii_map_text = VII_MAP_PATH.read_text(encoding='utf-8', errors='ignore') if VII_MAP_PATH.exists() else ''

print('Config ready. Output:', OUTPUT_DIR)
print('Run profile:', RUN_PROFILE)
print('Pass-1 model:', MODEL_PASS1)
print('Pass-2 model:', MODEL_PASS2)
print('Section II schema loaded:', bool(schema_text))
print('Section II governance loaded:', bool(gov_text))
print('Section II tradeoff governance loaded:', bool(ii_tradeoff_text))
print('Section II metric-eval governance loaded:', bool(ii_eval_text))
print('Section V axis loaded:', bool(v_axis_text))
print('Section V mapping loaded:', bool(v_map_text))
print('Section VI axis loaded:', bool(vi_axis_text))
print('Section VI mapping loaded:', bool(vi_map_text))
print('Section VII axis loaded:', bool(vii_axis_text))
print('Section VII mapping loaded:', bool(vii_map_text))


Config ready. Output: analysis/VIII_ev_v1
Run profile: FULL_RESCAN
Pass-1 model: meta-llama/llama-4-scout-17b-16e-instruct
Pass-2 model: llama-3.3-70b-versatile
Section II schema loaded: True
Section II governance loaded: True
Section II tradeoff governance loaded: True
Section II metric-eval governance loaded: True
Section V axis loaded: True
Section V mapping loaded: True
Section VI axis loaded: True
Section VI mapping loaded: True
Section VII axis loaded: True
Section VII mapping loaded: True


In [4]:
# @title 4. Load O_ISAC JSON Index
def load_json_index(json_dir: Path):
    index = {}
    for p in sorted(json_dir.glob('O_ISAC_*_v4.json')):
        paper_id = p.stem.replace('_v4','')
        try:
            index[paper_id] = json.loads(p.read_text(encoding='utf-8', errors='ignore'))
        except Exception as e:
            index[paper_id] = {'_error': str(e)}
    return index

def load_unified_csv(path: Path):
    if not path.exists():
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except Exception:
        return pd.DataFrame()

json_index = load_json_index(JSON_DIR)
unified_df = load_unified_csv(UNIFIED_CSV)
print('JSON files loaded:', len(json_index))
print('Unified CSV rows:', len(unified_df))


JSON files loaded: 221
Unified CSV rows: 224


In [5]:
# @title 5. Load Processed Markdowns (Canonical per paper)
def canonical_md_path(paths, paper_id):
    scored = []
    for p in paths:
        p = Path(p)
        score = 0
        if (p.parent / 'visual_analysis.txt').exists():
            score += 3
        if p.parent.name == paper_id and p.parent.parent.name == paper_id:
            score += 2
        score += len(p.parts) * 0.1
        scored.append((score, p))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[0][1] if scored else None

def load_processed_markdowns(target_ids=None, limit=None):
    search_path = PROCESSED_MD_DIR
    all_files = list(search_path.rglob('*.md'))
    md_files = [p for p in all_files if 'O_ISAC_' in p.name]

    grouped = {}
    for p in md_files:
        m = re.search(r'(O_ISAC_\d+)', p.name)
        if not m:
            continue
        paper_id = m.group(1)
        if target_ids and paper_id not in target_ids:
            continue
        grouped.setdefault(paper_id, []).append(p)

    records = []
    for i, (paper_id, paths) in enumerate(sorted(grouped.items())):
        if limit and i >= limit:
            break
        canon = canonical_md_path(paths, paper_id)
        if not canon:
            continue
        text = canon.read_text(encoding='utf-8', errors='ignore')
        lines = text.splitlines()
        va_path = canon.parent / 'visual_analysis.txt'
        va_text = va_path.read_text(encoding='utf-8', errors='ignore') if va_path.exists() else ''
        records.append({
            'paper_id': paper_id,
            'md_path': str(canon),
            'text': text,
            'lines': lines,
            'visual_analysis': va_text
        })
    return records

papers = load_processed_markdowns(target_ids=TARGET_PAPERS, limit=LIMIT)
print('Markdown papers loaded:', len(papers))


Markdown papers loaded: 221


In [6]:
# @title 6. Heading + Challenge Helpers
import re

MEDIUM_ALIAS_MAP = {
    'visible_light': 'wireless_vlc',
    'vlc': 'wireless_vlc',
    'rf': 'wireless_rf',
    'photo_thz': 'terahertz',
    'photonic_thz': 'terahertz',
}

def build_heading_map(lines):
    current = []
    heading_map = {}
    for i, line in enumerate(lines):
        if line.startswith('#'):
            level = len(line) - len(line.lstrip('#'))
            title = line.strip('#').strip()
            if level <= len(current):
                current = current[:level - 1]
            current.append(title)
        heading_map[i] = ' > '.join(current) if current else 'no_heading'
    return heading_map

def get_context(lines, idx, window=2):
    start = max(0, idx - window)
    end = min(len(lines), idx + window + 1)
    return '\n'.join(lines[start:end])

def normalize_token(value):
    s = str(value or '').strip().lower()
    if not s or s in {'nr', 'not reported', 'nan', 'none', 'na', 'n/a'}:
        return 'unknown'
    s = s.replace('/', '_').replace('&', ' and ')
    s = re.sub(r'[^a-z0-9_\-\s]+', '', s)
    s = re.sub(r'[\s\-]+', '_', s)
    s = re.sub(r'_+', '_', s).strip('_')
    return s or 'unknown'

def normalize_medium_label(value):
    s = normalize_token(value)
    return MEDIUM_ALIAS_MAP.get(s, s)

def get_record_medium(record):
    if not isinstance(record, dict):
        return 'unknown'
    clsf = record.get('study_level', {}).get('classification', {})
    if not isinstance(clsf, dict):
        return 'unknown'
    return normalize_medium_label(clsf.get('oisac_medium_class', 'unknown'))

def read_ids_from_csv(path):
    p = Path(path)
    if not p.exists():
        return set()
    try:
        d = pd.read_csv(p)
    except Exception:
        return set()
    if 'paper_id' not in d.columns:
        return set()
    return set(d['paper_id'].astype(str))

S5_SIGNAL_IDS = set()
for p in [
    'analysis/V_ev_v2/section5A_evidence.csv',
    'analysis/V_ev_v2/s5c_trade_mnts.csv',
    'analysis/V_ev_v2/section5C_tradeoff_points.csv',
]:
    S5_SIGNAL_IDS |= read_ids_from_csv(p)

S6_SIGNAL_IDS = set()
for p in [
    'analysis/VI_ev_v2/section6A_evidence.csv',
    'analysis/VI_ev_v2/section6B_opa_metrics.csv',
    'analysis/VI_ev_v2/section6C_ris_metrics.csv',
    'analysis/VI_ev_v2/section6D_evidence.csv',
]:
    S6_SIGNAL_IDS |= read_ids_from_csv(p)

S7_SIGNAL_IDS = set()
for p in [
    'analysis/VII_ev_v2/section7A_evidence.csv',
    'analysis/VII_ev_v2/section7B_evidence.csv',
    'analysis/VII_ev_v2/section7C_evidence.csv',
    'analysis/VII_ev_v2/section7D_evidence.csv',
    'analysis/VII_ev_v2/section7E_evidence.csv',
]:
    S7_SIGNAL_IDS |= read_ids_from_csv(p)

def get_upstream_signals(paper_id):
    pid = str(paper_id)
    return {
        'has_section5_signal': pid in S5_SIGNAL_IDS,
        'has_section6_signal': pid in S6_SIGNAL_IDS,
        'has_section7_signal': pid in S7_SIGNAL_IDS,
    }

def challenge_upstream_match(challenge_domain, signals):
    if challenge_domain == 'hardware_scalability_efficiency':
        return signals['has_section6_signal']
    if challenge_domain == 'channel_modeling_evaluation':
        return signals['has_section5_signal'] or signals['has_section7_signal']
    if challenge_domain == 'security_privacy_reliability':
        return signals['has_section7_signal']
    return signals['has_section5_signal'] or signals['has_section6_signal'] or signals['has_section7_signal']

print('Section5 signal papers:', len(S5_SIGNAL_IDS))
print('Section6 signal papers:', len(S6_SIGNAL_IDS))
print('Section7 signal papers:', len(S7_SIGNAL_IDS))


Section5 signal papers: 221
Section6 signal papers: 221
Section7 signal papers: 221


In [7]:
# @title 7. Groq Client + Variant Generator (Cache + Per-Model Rate Limit)
from groq import Groq
from collections import deque
import time
import random

VARIANT_CACHE = OUTPUT_DIR / 'variant_cache.json'
if VARIANT_CACHE.exists():
    variant_cache = json.loads(VARIANT_CACHE.read_text(encoding='utf-8'))
else:
    variant_cache = {}

_GROQ_CLIENT = None
REQUEST_LOG_BY_MODEL = {}


def get_groq_client():
    global _GROQ_CLIENT
    if _GROQ_CLIENT is not None:
        return _GROQ_CLIENT

    try:
        api_key = userdata.get('GROQ_API_KEY')
    except Exception:
        api_key = os.environ.get('GROQ_API_KEY')

    if not api_key:
        raise ValueError('GROQ_API_KEY not found in Colab Secrets or env.')

    _GROQ_CLIENT = Groq(api_key=api_key)
    return _GROQ_CLIENT


def get_model_rpm(model_name):
    return RPM_BY_MODEL.get(model_name, DEFAULT_RPM)


def throttle_requests(model_name):
    rpm = get_model_rpm(model_name)
    if rpm <= 0:
        return

    q = REQUEST_LOG_BY_MODEL.setdefault(model_name, deque())
    now = time.time()

    while q and now - q[0] > 60:
        q.popleft()

    if len(q) >= rpm:
        wait_s = 60 - (now - q[0]) + 0.1
        wait_s = max(wait_s, 0.1)
        print(f'Rate limit guard ({model_name}): sleeping {wait_s:.1f}s')
        time.sleep(wait_s)
        now = time.time()
        while q and now - q[0] > 60:
            q.popleft()

    q.append(time.time())


def safe_chat_completion(model_name, messages, expect_json=False, temperature=0.2):
    if not LLM_CALLS:
        return None

    client = get_groq_client()

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            throttle_requests(model_name)
            kwargs = {
                'model': model_name,
                'messages': messages,
                'temperature': temperature
            }
            if expect_json:
                kwargs['response_format'] = {'type': 'json_object'}

            resp = client.chat.completions.create(**kwargs)
            return resp.choices[0].message.content
        except Exception as e:
            if attempt >= MAX_RETRIES:
                print(f'LLM call failed ({model_name}) after {MAX_RETRIES} attempts: {e}')
                return None
            sleep_s = RETRY_BASE_SECONDS * (2 ** (attempt - 1)) + random.uniform(0.0, 0.5)
            print(f'LLM retry ({model_name}) {attempt}/{MAX_RETRIES}: {e}; sleeping {sleep_s:.1f}s')
            time.sleep(sleep_s)


def get_variants(concept):
    if concept in variant_cache:
        vals = variant_cache[concept]
        return vals[:MAX_VARIANTS_PER_CONCEPT]

    if not LLM_CALLS:
        vals = [concept]
        variant_cache[concept] = vals
        return vals

    system_prompt = (
        'You generate lexical variants and paraphrases for evidence retrieval. '
        'Return compact JSON: {"variants": ["..."]}.'
    )
    user_prompt = (
        f'Concept: {concept}\n'
        'Return up to 12 variants including synonyms, abbreviations, paraphrases, and morphological forms. '
        'Keep each variant short.'
    )

    content = safe_chat_completion(
        model_name=MODEL_VARIANT_GEN,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt}
        ],
        expect_json=True,
        temperature=0.2
    )

    variants = [concept]
    if content:
        try:
            data = json.loads(content)
            llm_vars = data.get('variants', [])
            if isinstance(llm_vars, list):
                for item in llm_vars:
                    if isinstance(item, str) and item.strip():
                        variants.append(item.strip())
        except Exception:
            pass

    dedup = []
    seen = set()
    for v in variants:
        key = v.lower().strip()
        if not key or key in seen:
            continue
        seen.add(key)
        dedup.append(v)

    dedup = dedup[:MAX_VARIANTS_PER_CONCEPT]
    variant_cache[concept] = dedup
    VARIANT_CACHE.write_text(json.dumps(variant_cache, ensure_ascii=False, indent=2), encoding='utf-8')
    return dedup


In [8]:
# @title 8. Retrieval + Entailment Classification (Two-Model, Batched, Checkpointed)
def scan_lines_for_variants(lines, variants, fuzzy_threshold=85):
    hits = []
    for i, line in enumerate(lines):
        text = line.strip()
        if not text:
            continue
        low = text.lower()
        for v in variants:
            vlow = v.lower()
            if vlow in low:
                hits.append((i, line, v, 'lexical'))
                break
            score = fuzz.partial_ratio(vlow, low)
            if score >= fuzzy_threshold:
                hits.append((i, line, v, f'fuzzy:{score}'))
                break
    return hits


def clip_text(text, max_chars=MAX_CONTEXT_CHARS):
    if text is None:
        return ''
    text = str(text)
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + ' ...'


def chunk_list(items, n):
    for i in range(0, len(items), n):
        yield items[i:i+n]


def parse_batch_results(content, n):
    fallback = [{'label': 'WEAK', 'rationale': 'LLM parse failed'} for _ in range(n)]
    if not content:
        return fallback

    try:
        parsed = json.loads(content)
        results = parsed.get('results', [])
        mapped = {int(r['idx']): r for r in results if isinstance(r, dict) and 'idx' in r}
        out = []
        for i in range(n):
            r = mapped.get(i)
            if not r:
                out.append({'label': 'WEAK', 'rationale': 'No label'})
                continue
            label = str(r.get('label', 'WEAK')).upper().strip()
            if label not in {'DIRECT', 'INDIRECT', 'NONE'}:
                label = 'WEAK'
            out.append({'label': label, 'rationale': str(r.get('rationale', ''))})
        return out
    except Exception:
        return fallback


def classify_with_model(concept, contexts, model_name, hint_labels=None):
    compact = []
    for i, ctx in enumerate(contexts):
        row = {'idx': i, 'context': clip_text(ctx)}
        if hint_labels and i < len(hint_labels):
            row['hint_label'] = hint_labels[i]
        compact.append(row)

    system_prompt = (
        'You are an evidence auditor. '
        'For each snippet, decide if the concept is DIRECT, INDIRECT, or NONE. '
        'Return strict JSON object with key "results": '
        '[{"idx":0,"label":"DIRECT|INDIRECT|NONE","rationale":"..."}]'
    )
    user_prompt = (
        f'Concept: {concept}\n'
        f'Snippets JSON:\n{json.dumps(compact, ensure_ascii=False)}'
    )

    content = safe_chat_completion(
        model_name=model_name,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt}
        ],
        expect_json=True,
        temperature=0.1
    )

    return parse_batch_results(content, len(contexts))


def classify_hits_batch(concept, contexts):
    if not contexts:
        return []

    if not LLM_CALLS:
        return [{
            'label': 'WEAK',
            'rationale': 'LLM disabled',
            'label_pass1': 'WEAK',
            'rationale_pass1': 'LLM disabled',
            'model_pass1': MODEL_PASS1,
            'label_pass2': '',
            'rationale_pass2': '',
            'model_pass2': '',
            'escalated': False
        } for _ in contexts]

    pass1 = classify_with_model(concept, contexts, MODEL_PASS1)
    out = []
    for r in pass1:
        out.append({
            'label': r.get('label', 'WEAK'),
            'rationale': r.get('rationale', ''),
            'label_pass1': r.get('label', 'WEAK'),
            'rationale_pass1': r.get('rationale', ''),
            'model_pass1': MODEL_PASS1,
            'label_pass2': '',
            'rationale_pass2': '',
            'model_pass2': '',
            'escalated': False
        })

    if USE_ESCALATION:
        idxs = [i for i, r in enumerate(out) if r['label'] in ESCALATE_LABELS]
        if idxs:
            sub_contexts = [contexts[i] for i in idxs]
            hints = [out[i]['label_pass1'] for i in idxs]
            pass2 = classify_with_model(concept, sub_contexts, MODEL_PASS2, hint_labels=hints)

            for j, i in enumerate(idxs):
                r2 = pass2[j]
                out[i]['label_pass2'] = r2.get('label', 'WEAK')
                out[i]['rationale_pass2'] = r2.get('rationale', '')
                out[i]['model_pass2'] = MODEL_PASS2
                out[i]['escalated'] = True

                if r2.get('label') in {'DIRECT', 'INDIRECT', 'NONE'}:
                    out[i]['label'] = r2.get('label')
                    out[i]['rationale'] = r2.get('rationale', '')

    return out


def classify_hits_chunked(concept, contexts):
    out = []
    for chunk in chunk_list(contexts, CLASSIFY_CHUNK_SIZE):
        out.extend(classify_hits_batch(concept, chunk))
    return out


def append_rows_csv(out_csv, rows):
    if not rows:
        return
    df_new = pd.DataFrame(rows)
    if out_csv.exists():
        df_new.to_csv(out_csv, mode='a', header=False, index=False)
    else:
        df_new.to_csv(out_csv, index=False)


def checkpoint_path(section_name):
    return CHECKPOINT_DIR / f'{section_name}_done_ids.json'


def load_done_ids(section_name):
    if not RESUME:
        return set()
    cp = checkpoint_path(section_name)
    if not cp.exists():
        return set()
    try:
        data = json.loads(cp.read_text(encoding='utf-8'))
        return set(data)
    except Exception:
        return set()


def save_done_ids(section_name, done_ids):
    cp = checkpoint_path(section_name)
    cp.write_text(json.dumps(sorted(list(done_ids)), ensure_ascii=False, indent=2), encoding='utf-8')


def process_in_batches(records):
    for batch in chunk_list(records, BATCH_SIZE_PAPERS):
        yield batch


def llm_fields_from_cls(cls):
    return {
        'llm_model_pass1': cls.get('model_pass1', ''),
        'llm_label_pass1': cls.get('label_pass1', ''),
        'llm_model_pass2': cls.get('model_pass2', ''),
        'llm_label_pass2': cls.get('label_pass2', ''),
        'llm_escalated': cls.get('escalated', False),
    }


In [9]:
# @title 9. Section 8A Evidence (Standardization and Interoperability Challenges)

def run_section8_challenge_extraction(section_code, challenge_domain, lexical_terms):
    section_name = f'section8{section_code}'
    out_csv = OUTPUT_DIR / f'{section_name}_evidence.csv'
    done_ids = load_done_ids(section_name)
    pending = [p for p in papers if p['paper_id'] not in done_ids]
    print(f'{section_name}: pending papers = {len(pending)}')

    variants = []
    seen = set()
    for term in lexical_terms:
        for item in get_variants(term):
            key = str(item).strip().lower()
            if key and key not in seen:
                seen.add(key)
                variants.append(str(item).strip())
    variants = variants[:MAX_VARIANTS_PER_CONCEPT]

    for batch in process_in_batches(pending):
        batch_rows = []
        for paper in tqdm(batch, desc=f'{section_name} batch'):
            paper_id = paper['paper_id']
            lines = paper['lines']
            heading_map = build_heading_map(lines)
            record = json_index.get(paper_id, {})
            medium = get_record_medium(record)
            signals = get_upstream_signals(paper_id)

            hits = scan_lines_for_variants(lines, variants)[:MAX_HITS_PER_CONCEPT_PER_PAPER]
            contexts = [get_context(lines, idx) for idx, _, _, _ in hits]
            cls_all = classify_hits_chunked(challenge_domain, contexts)

            text_hit_count = 0
            for (hit, cls) in zip(hits, cls_all):
                idx, line, variant, match_type = hit
                text_hit_count += 1
                batch_rows.append({
                    'paper_id': paper_id,
                    'section': f'8{section_code}',
                    'challenge_domain': challenge_domain,
                    'variant': variant,
                    'match_type': match_type,
                    'strength': cls.get('label', 'WEAK'),
                    'rationale': cls.get('rationale', ''),
                    'quote': line.strip(),
                    'line_start': idx + 1,
                    'line_end': idx + 1,
                    'heading_path': heading_map.get(idx, 'no_heading'),
                    'json_path': '',
                    'json_value': '',
                    'medium': medium,
                    'has_section5_signal': signals['has_section5_signal'],
                    'has_section6_signal': signals['has_section6_signal'],
                    'has_section7_signal': signals['has_section7_signal'],
                    **llm_fields_from_cls(cls),
                })

            if text_hit_count == 0 and challenge_upstream_match(challenge_domain, signals):
                linked = []
                if signals['has_section5_signal']:
                    linked.append('S5')
                if signals['has_section6_signal']:
                    linked.append('S6')
                if signals['has_section7_signal']:
                    linked.append('S7')
                batch_rows.append({
                    'paper_id': paper_id,
                    'section': f'8{section_code}',
                    'challenge_domain': challenge_domain,
                    'variant': '',
                    'match_type': 'upstream_bridge',
                    'strength': 'INDIRECT',
                    'rationale': f'Cross-section bridge from {";".join(linked)} outputs.',
                    'quote': '',
                    'line_start': '',
                    'line_end': '',
                    'heading_path': '',
                    'json_path': 'section5_6_7_bridge',
                    'json_value': ';'.join(linked),
                    'medium': medium,
                    'has_section5_signal': signals['has_section5_signal'],
                    'has_section6_signal': signals['has_section6_signal'],
                    'has_section7_signal': signals['has_section7_signal'],
                    'llm_model_pass1': 'bridge',
                    'llm_label_pass1': 'INDIRECT',
                    'llm_model_pass2': '',
                    'llm_label_pass2': '',
                    'llm_escalated': False,
                })

            done_ids.add(paper_id)

        append_rows_csv(out_csv, batch_rows)
        save_done_ids(section_name, done_ids)
        print(f'{section_name}: wrote {len(batch_rows)} rows; done={len(done_ids)}')

    print('Saved:', out_csv)

terms_8A = [
    'standardization', 'interoperability', 'protocol harmonization', 'reference architecture',
    'cross-vendor compatibility', 'benchmark protocol', 'compliance framework',
    'standard gap', 'lack of standards', 'heterogeneous integration'
]
run_section8_challenge_extraction('A', 'standardization_interoperability', terms_8A)


section8A: pending papers = 221


section8A batch: 100%|██████████| 10/10 [00:07<00:00,  1.29it/s]


section8A: wrote 19 rows; done=10


section8A batch: 100%|██████████| 10/10 [00:07<00:00,  1.35it/s]


section8A: wrote 19 rows; done=20


...

Rate limit guard (llama-3.3-70b-versatile): sleeping 0.4s


section8A batch: 100%|██████████| 10/10 [00:04<00:00,  2.46it/s]


section8A: wrote 10 rows; done=220


section8A batch: 100%|██████████| 1/1 [00:00<00:00, 56.61it/s]

section8A: wrote 1 rows; done=221
Saved: analysis/VIII_ev_v1/section8A_evidence.csv


In [10]:
# @title 10. Section 8B Evidence (Hardware Scalability, Cost, and Energy Challenges)
terms_8B = [
    'hardware complexity', 'integration complexity', 'cost overhead', 'power consumption',
    'energy efficiency', 'scalability bottleneck', 'packaging loss', 'thermal drift',
    'calibration challenge', 'miniaturization tradeoff'
]
run_section8_challenge_extraction('B', 'hardware_scalability_efficiency', terms_8B)


section8B: pending papers = 221


section8B batch: 100%|██████████| 10/10 [00:01<00:00,  5.39it/s]


section8B: wrote 10 rows; done=10


section8B batch: 100%|██████████| 10/10 [00:04<00:00,  2.23it/s]


section8B: wrote 11 rows; done=20


...

Rate limit guard (llama-3.3-70b-versatile): sleeping 0.3s


section8B batch: 100%|██████████| 10/10 [00:06<00:00,  1.59it/s]


section8B: wrote 12 rows; done=220


section8B batch: 100%|██████████| 1/1 [00:00<00:00, 30.65it/s]

section8B: wrote 1 rows; done=221
Saved: analysis/VIII_ev_v1/section8B_evidence.csv


In [11]:
# @title 11. Section 8C Evidence (Channel Modeling and Evaluation Challenges)
terms_8C = [
    'channel modeling gap', 'propagation model mismatch', 'dataset scarcity', 'benchmark scarcity',
    'evaluation inconsistency', 'reproducibility challenge', 'domain shift', 'measurement uncertainty',
    'lack of public dataset', 'cross-medium generalization'
]
run_section8_challenge_extraction('C', 'channel_modeling_evaluation', terms_8C)


section8C: pending papers = 221


section8C batch:  50%|█████     | 5/10 [00:04<00:04,  1.14it/s]

Rate limit guard (llama-3.3-70b-versatile): sleeping 6.8s


section8C batch: 100%|██████████| 10/10 [00:12<00:00,  1.27s/it]


section8C: wrote 22 rows; done=10


...

Rate limit guard (llama-3.3-70b-versatile): sleeping 0.2s
Rate limit guard (llama-3.3-70b-versatile): sleeping 0.1s


section8C batch: 100%|██████████| 10/10 [00:04<00:00,  2.48it/s]


section8C: wrote 18 rows; done=220


section8C batch: 100%|██████████| 1/1 [00:02<00:00,  2.13s/it]

section8C: wrote 6 rows; done=221
Saved: analysis/VIII_ev_v1/section8C_evidence.csv


In [12]:
# @title 12. Section 8D Evidence (Security, Privacy, Reliability, Safety Challenges)
terms_8D = [
    'security threat', 'privacy risk', 'eavesdropping', 'spoofing attack', 'adversarial attack',
    'robustness limitation', 'reliability issue', 'safety-critical constraint',
    'fault tolerance limitation', 'trustworthiness challenge'
]
run_section8_challenge_extraction('D', 'security_privacy_reliability', terms_8D)


section8D: pending papers = 221


section8D batch: 100%|██████████| 10/10 [00:00<00:00, 78.10it/s]


section8D: wrote 10 rows; done=10


section8D batch: 100%|██████████| 10/10 [00:01<00:00,  6.48it/s]


section8D: wrote 10 rows; done=20


...

section8D: wrote 15 rows; done=210


section8D batch: 100%|██████████| 10/10 [00:00<00:00, 80.22it/s]


section8D: wrote 10 rows; done=220


section8D batch: 100%|██████████| 1/1 [00:00<00:00, 51.69it/s]

section8D: wrote 1 rows; done=221
Saved: analysis/VIII_ev_v1/section8D_evidence.csv


In [13]:
# @title 13. Section 8E Evidence + Section 8F Research Roadmap Outputs
terms_8E = [
    'deployment barrier', 'technology readiness', 'field trial limitation', 'rf-optical convergence',
    'hybrid deployment', 'migration strategy', 'future research direction', 'open challenge',
    'open problem', 'research roadmap'
]
run_section8_challenge_extraction('E', 'deployment_convergence_roadmap', terms_8E)

a_csv = OUTPUT_DIR / 'section8A_evidence.csv'
b_csv = OUTPUT_DIR / 'section8B_evidence.csv'
c_csv = OUTPUT_DIR / 'section8C_evidence.csv'
d_csv = OUTPUT_DIR / 'section8D_evidence.csv'
e_csv = OUTPUT_DIR / 'section8E_evidence.csv'

A = pd.read_csv(a_csv) if a_csv.exists() else pd.DataFrame()
B = pd.read_csv(b_csv) if b_csv.exists() else pd.DataFrame()
C = pd.read_csv(c_csv) if c_csv.exists() else pd.DataFrame()
D = pd.read_csv(d_csv) if d_csv.exists() else pd.DataFrame()
E = pd.read_csv(e_csv) if e_csv.exists() else pd.DataFrame()

challenge_to_df = {
    'standardization_interoperability': A,
    'hardware_scalability_efficiency': B,
    'channel_modeling_evaluation': C,
    'security_privacy_reliability': D,
    'deployment_convergence_roadmap': E,
}

def strict_supported_ids(df):
    if df.empty or 'paper_id' not in df.columns:
        return set()
    out = set()
    for pid, grp in df.groupby('paper_id'):
        direct = (grp['strength'].astype(str).str.upper() == 'DIRECT').sum() if 'strength' in grp.columns else 0
        indirect = (grp['strength'].astype(str).str.upper() == 'INDIRECT').sum() if 'strength' in grp.columns else 0
        if direct >= 1 or indirect >= 2:
            out.add(str(pid))
    return out

all_paper_ids = sorted([p.get('paper_id') for p in papers])
challenge_supported = {c: strict_supported_ids(df) for c, df in challenge_to_df.items()}

paper_rows = []
for pid in all_paper_ids:
    rec = json_index.get(pid, {})
    medium = get_record_medium(rec)
    sig = get_upstream_signals(pid)
    row = {'paper_id': pid, 'medium': medium, **sig}
    n_supported = 0
    for c in challenge_to_df.keys():
        has_val = pid in challenge_supported[c]
        row[f'has_{c}'] = has_val
        if has_val:
            n_supported += 1
    row['n_supported_challenges'] = n_supported
    paper_rows.append(row)

paper_df = pd.DataFrame(paper_rows)
paper_map_csv = OUTPUT_DIR / 's8f_pap_chal_map.csv'
paper_df.to_csv(paper_map_csv, index=False)

agenda_rows = []
for c, df in challenge_to_df.items():
    sids = challenge_supported[c]
    sub = paper_df[paper_df['paper_id'].isin(sids)] if not paper_df.empty else pd.DataFrame()
    direct_rows = int((df['strength'].astype(str).str.upper() == 'DIRECT').sum()) if not df.empty else 0
    indirect_rows = int((df['strength'].astype(str).str.upper() == 'INDIRECT').sum()) if not df.empty else 0
    n_mediums = int(sub['medium'].nunique()) if not sub.empty else 0
    linked_s5 = int(sub['has_section5_signal'].sum()) if not sub.empty else 0
    linked_s6 = int(sub['has_section6_signal'].sum()) if not sub.empty else 0
    linked_s7 = int(sub['has_section7_signal'].sum()) if not sub.empty else 0
    priority_score = int(len(sids) + 2 * n_mediums + linked_s5 + linked_s6 + linked_s7)
    agenda_rows.append({
        'challenge_domain': c,
        'n_supported_papers': int(len(sids)),
        'n_direct_rows': direct_rows,
        'n_indirect_rows': indirect_rows,
        'n_mediums': n_mediums,
        'linked_section5_papers': linked_s5,
        'linked_section6_papers': linked_s6,
        'linked_section7_papers': linked_s7,
        'priority_score': priority_score,
    })

agenda_df = pd.DataFrame(agenda_rows).sort_values(['priority_score', 'n_supported_papers'], ascending=False)
agenda_csv = OUTPUT_DIR / 'section8F_research_agenda.csv'
agenda_df.to_csv(agenda_csv, index=False)
summary_table_csv = OUTPUT_DIR / 'section8F_summary_table.csv'
agenda_df[['challenge_domain', 'n_supported_papers', 'n_mediums', 'priority_score']].to_csv(summary_table_csv, index=False)

dep_csv = OUTPUT_DIR / 's8f_dep_cov.csv'
agenda_df[['challenge_domain', 'linked_section5_papers', 'linked_section6_papers', 'linked_section7_papers']].to_csv(dep_csv, index=False)

summary_payload = {
    'n_total_papers': int(len(all_paper_ids)),
    'n_standardization_interoperability_papers': int(len(challenge_supported['standardization_interoperability'])),
    'n_hardware_scalability_efficiency_papers': int(len(challenge_supported['hardware_scalability_efficiency'])),
    'n_channel_modeling_evaluation_papers': int(len(challenge_supported['channel_modeling_evaluation'])),
    'n_security_privacy_reliability_papers': int(len(challenge_supported['security_privacy_reliability'])),
    'n_deployment_convergence_roadmap_papers': int(len(challenge_supported['deployment_convergence_roadmap'])),
    'n_multi_challenge_papers': int((paper_df['n_supported_challenges'] >= 2).sum()) if not paper_df.empty else 0,
    'n_all_challenge_domains': int(len(challenge_to_df)),
}
summary_json = OUTPUT_DIR / 'section8F_summary.json'
summary_json.write_text(json.dumps(summary_payload, indent=2), encoding='utf-8')

print('Saved:', agenda_csv)
print('Saved:', summary_table_csv)
print('Saved:', dep_csv)
print('Saved:', paper_map_csv)
print('Saved:', summary_json)


section8E: pending papers = 221


section8E batch: 100%|██████████| 10/10 [00:01<00:00,  9.12it/s]


section8E: wrote 13 rows; done=10


section8E batch: 100%|██████████| 10/10 [00:00<00:00, 44.52it/s]


section8E: wrote 10 rows; done=20


...

section8E: wrote 10 rows; done=210


section8E batch: 100%|██████████| 10/10 [00:00<00:00, 72.36it/s]


section8E: wrote 10 rows; done=220


section8E batch: 100%|██████████| 1/1 [00:00<00:00, 53.61it/s]


section8E: wrote 1 rows; done=221
Saved: analysis/VIII_ev_v1/section8E_evidence.csv
Saved: analysis/VIII_ev_v1/section8F_research_agenda.csv
Saved: analysis/VIII_ev_v1/section8F_summary_table.csv
Saved: analysis/VIII_ev_v1/s8f_dep_cov.csv
Saved: analysis/VIII_ev_v1/s8f_pap_chal_map.csv
Saved: analysis/VIII_ev_v1/section8F_summary.json


In [14]:
# @title 14. Post-Processing Artifacts (Section 8)
import hashlib
from collections import defaultdict

def as_int_or_blank(x):
    try:
        if pd.isna(x) or x == '':
            return ''
        return int(float(x))
    except Exception:
        return ''

a_csv = OUTPUT_DIR / 'section8A_evidence.csv'
b_csv = OUTPUT_DIR / 'section8B_evidence.csv'
c_csv = OUTPUT_DIR / 'section8C_evidence.csv'
d_csv = OUTPUT_DIR / 'section8D_evidence.csv'
e_csv = OUTPUT_DIR / 'section8E_evidence.csv'

A = pd.read_csv(a_csv) if a_csv.exists() else pd.DataFrame()
B = pd.read_csv(b_csv) if b_csv.exists() else pd.DataFrame()
C = pd.read_csv(c_csv) if c_csv.exists() else pd.DataFrame()
D = pd.read_csv(d_csv) if d_csv.exists() else pd.DataFrame()
E = pd.read_csv(e_csv) if e_csv.exists() else pd.DataFrame()
frames = [('8A', A), ('8B', B), ('8C', C), ('8D', D), ('8E', E)]

retrieval_path = OUTPUT_DIR / 'retrieval_hits.jsonl'
with retrieval_path.open('w', encoding='utf-8') as f:
    for sec_name, df in frames:
        if df.empty:
            continue
        for _, r in df.iterrows():
            rec = {
                'paper_id': str(r.get('paper_id', '')),
                'section': sec_name,
                'challenge_domain': str(r.get('challenge_domain', '')),
                'variant': str(r.get('variant', '')),
                'match_type': str(r.get('match_type', '')),
                'quote': str(r.get('quote', '')),
                'line_start': as_int_or_blank(r.get('line_start', '')),
                'line_end': as_int_or_blank(r.get('line_end', '')),
                'heading_path': str(r.get('heading_path', '')),
                'strength': str(r.get('strength', '')),
                'rationale': str(r.get('rationale', '')),
            }
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')
print('Saved:', retrieval_path)

anchor_rows = []
for sec_name, df in frames:
    if df.empty:
        continue
    for _, r in df.iterrows():
        challenge = str(r.get('challenge_domain', '')).strip()
        claim_key = f'{sec_name}|{challenge}'
        claim_id = hashlib.sha1(claim_key.encode('utf-8')).hexdigest()[:12]
        anchor_rows.append({
            'claim_id': claim_id,
            'claim_key': claim_key,
            'section': sec_name,
            'paper_id': str(r.get('paper_id', '')),
            'challenge_domain': challenge,
            'variant': str(r.get('variant', '')),
            'match_type': str(r.get('match_type', '')),
            'strength': str(r.get('strength', '')),
            'rationale': str(r.get('rationale', '')),
            'quote': str(r.get('quote', '')),
            'line_start': as_int_or_blank(r.get('line_start', '')),
            'line_end': as_int_or_blank(r.get('line_end', '')),
            'heading_path': str(r.get('heading_path', '')),
            'json_path': str(r.get('json_path', '')),
            'json_value': str(r.get('json_value', '')),
            'has_section5_signal': bool(r.get('has_section5_signal', False)),
            'has_section6_signal': bool(r.get('has_section6_signal', False)),
            'has_section7_signal': bool(r.get('has_section7_signal', False)),
        })

anchor_cols = ['claim_id','claim_key','section','paper_id','challenge_domain','variant','match_type','strength','rationale','quote','line_start','line_end','heading_path','json_path','json_value','has_section5_signal','has_section6_signal','has_section7_signal','claim_supported']
if anchor_rows:
    anc = pd.DataFrame(anchor_rows)
    agg = anc.assign(
        is_direct=anc['strength'].astype(str).str.upper().eq('DIRECT'),
        is_indirect=anc['strength'].astype(str).str.upper().eq('INDIRECT')
    ).groupby('claim_id', as_index=False)[['is_direct', 'is_indirect']].sum()
    agg['claim_supported'] = (agg['is_direct'] >= 1) | (agg['is_indirect'] >= 2)
    anc = anc.merge(agg[['claim_id', 'claim_supported']], on='claim_id', how='left')
else:
    anc = pd.DataFrame(columns=anchor_cols)
anc = anc[anchor_cols]
anchor_path = OUTPUT_DIR / 'anchor_table.csv'
anc.to_csv(anchor_path, index=False)
print('Saved:', anchor_path)

supported_by_paper = defaultdict(set)
if not anc.empty:
    for _, r in anc.iterrows():
        if bool(r.get('claim_supported', False)):
            supported_by_paper[str(r.get('paper_id', ''))].add(str(r.get('challenge_domain', '')))

graph_path = OUTPUT_DIR / 'evidence_graph.jsonl'
cluster_rows = []
with graph_path.open('w', encoding='utf-8') as f:
    for paper_id, rec in sorted(json_index.items()):
        medium = get_record_medium(rec)
        concepts = supported_by_paper.get(paper_id, set())
        sig = get_upstream_signals(paper_id)
        graph_rec = {
            'paper_id': paper_id,
            'structured': {
                'medium': medium,
                'has_standardization_interoperability_supported': 'standardization_interoperability' in concepts,
                'has_hardware_scalability_efficiency_supported': 'hardware_scalability_efficiency' in concepts,
                'has_channel_modeling_evaluation_supported': 'channel_modeling_evaluation' in concepts,
                'has_security_privacy_reliability_supported': 'security_privacy_reliability' in concepts,
                'has_deployment_convergence_roadmap_supported': 'deployment_convergence_roadmap' in concepts,
                'has_section5_signal': sig['has_section5_signal'],
                'has_section6_signal': sig['has_section6_signal'],
                'has_section7_signal': sig['has_section7_signal'],
            },
            'anchor_count': len([a for a in anchor_rows if a['paper_id'] == paper_id]),
        }
        f.write(json.dumps(graph_rec, ensure_ascii=False) + '\n')
        cluster_rows.append({'paper_id': paper_id, 'medium': medium, **graph_rec['structured'], 'anchor_count': graph_rec['anchor_count']})

cluster_path = OUTPUT_DIR / 'cluster_map.csv'
pd.DataFrame(cluster_rows).to_csv(cluster_path, index=False)
print('Saved:', graph_path)
print('Saved:', cluster_path)

axis_md = '\n'.join([
    '# Section 8 Axis Definitions (v1)',
    '',
    'Axis-1 Medium: normalized labels aligned with Section IV/Section V policies.',
    'Axis-2 Challenge domains: standardization_interoperability, hardware_scalability_efficiency, channel_modeling_evaluation, security_privacy_reliability, deployment_convergence_roadmap.',
    'Axis-3 Cross-section links: Section V (tradeoff), Section VI (enablers), Section VII (applications).',
    'Axis-4 Evidence gate: claim_supported = (>=1 DIRECT) OR (>=2 INDIRECT).',
    'Governance note: Section VIII must preserve Section II plane and metric alias rules.',
])
(OUTPUT_DIR / 'axis_definitions.md').write_text(axis_md, encoding='utf-8')

mapping_md = '\n'.join([
    '# Section 8 Mapping Rules (v1)',
    '',
    '1. Challenge claims are text-anchored first, cross-section bridges second.',
    '2. Upstream bridge rows (Section5/6/7) are INDIRECT by design and cannot replace DIRECT textual evidence.',
    '3. Section V tradeoff evidence informs challenge prioritization.',
    '4. Section VI enabler constraints inform hardware/deployment challenge narratives.',
    '5. Section VII application gaps inform roadmap risks.',
    '6. Any OSNR/SNR sentence must preserve Section II plane-separation wording.',
])
(OUTPUT_DIR / 'mapping_rules.md').write_text(mapping_md, encoding='utf-8')
print('Saved:', OUTPUT_DIR / 'axis_definitions.md')
print('Saved:', OUTPUT_DIR / 'mapping_rules.md')

violations = []
seen = set()
for sec_name, df in frames:
    if df.empty:
        continue
    text_df = df[df['match_type'].astype(str).str.lower() != 'upstream_bridge'].copy() if 'match_type' in df.columns else df.copy()
    for pid, grp in text_df.groupby('paper_id'):
        direct = (grp['strength'].astype(str).str.upper() == 'DIRECT').sum() if 'strength' in grp.columns else 0
        indirect = (grp['strength'].astype(str).str.upper() == 'INDIRECT').sum() if 'strength' in grp.columns else 0
        if direct < 1 and indirect < 2:
            challenge = str(grp['challenge_domain'].iloc[0]) if 'challenge_domain' in grp.columns else 'unknown'
            key = (str(pid), 'EVIDENCE_WEAK', challenge)
            if key not in seen:
                seen.add(key)
                violations.append({'paper_id': str(pid), 'section': sec_name, 'category': 'EVIDENCE_WEAK', 'severity': 'MINOR', 'reason': f'{challenge} lacks support gate (text anchors)', 'evidence': f'direct={direct}; indirect={indirect}'})

viol_path = OUTPUT_DIR / 'contract_violations.csv'
pd.DataFrame(violations, columns=['paper_id', 'section', 'category', 'severity', 'reason', 'evidence']).to_csv(viol_path, index=False)
print('Saved:', viol_path, 'rows=', len(violations))


Saved: analysis/VIII_ev_v1/retrieval_hits.jsonl
Saved: analysis/VIII_ev_v1/anchor_table.csv
Saved: analysis/VIII_ev_v1/evidence_graph.jsonl
Saved: analysis/VIII_ev_v1/cluster_map.csv
Saved: analysis/VIII_ev_v1/axis_definitions.md
Saved: analysis/VIII_ev_v1/mapping_rules.md
Saved: analysis/VIII_ev_v1/contract_violations.csv rows= 242


In [15]:
# @title 15. Section 8G Cross-Section Alignment (Section5/6/7 vs Section8 Challenges)
challenge_files = {
    'standardization_interoperability': OUTPUT_DIR / 'section8A_evidence.csv',
    'hardware_scalability_efficiency': OUTPUT_DIR / 'section8B_evidence.csv',
    'channel_modeling_evaluation': OUTPUT_DIR / 'section8C_evidence.csv',
    'security_privacy_reliability': OUTPUT_DIR / 'section8D_evidence.csv',
    'deployment_convergence_roadmap': OUTPUT_DIR / 'section8E_evidence.csv',
}

def strict_supported_ids(df):
    if df.empty or 'paper_id' not in df.columns:
        return set()
    out = set()
    for pid, grp in df.groupby('paper_id'):
        direct = (grp['strength'].astype(str).str.upper() == 'DIRECT').sum() if 'strength' in grp.columns else 0
        indirect = (grp['strength'].astype(str).str.upper() == 'INDIRECT').sum() if 'strength' in grp.columns else 0
        if direct >= 1 or indirect >= 2:
            out.add(str(pid))
    return out

rows = []
example_rows = []
max_show = 30
for challenge, path in challenge_files.items():
    df = pd.read_csv(path) if path.exists() else pd.DataFrame()
    strict_ids = strict_supported_ids(df)
    s5_ids = set()
    s6_ids = set()
    s7_ids = set()
    if not df.empty and 'paper_id' in df.columns:
        sub = df[df['paper_id'].astype(str).isin(strict_ids)]
        if 'has_section5_signal' in sub.columns:
            s5_ids = set(sub[sub['has_section5_signal'].astype(bool)]['paper_id'].astype(str))
        if 'has_section6_signal' in sub.columns:
            s6_ids = set(sub[sub['has_section6_signal'].astype(bool)]['paper_id'].astype(str))
        if 'has_section7_signal' in sub.columns:
            s7_ids = set(sub[sub['has_section7_signal'].astype(bool)]['paper_id'].astype(str))
    union_upstream = s5_ids | s6_ids | s7_ids
    rows.append({
        'challenge_domain': challenge,
        'strict_evidence_count': len(strict_ids),
        'linked_section5_count': len(s5_ids),
        'linked_section6_count': len(s6_ids),
        'linked_section7_count': len(s7_ids),
        'linked_any_upstream_count': len(union_upstream),
        'strict_without_upstream_count': len(strict_ids - union_upstream),
    })
    example_rows.append({'challenge_domain': challenge, 'group': 'strict_without_upstream', 'paper_ids': ';'.join(sorted(list(strict_ids - union_upstream))[:max_show])})

cmp_df = pd.DataFrame(rows)
cmp_csv = OUTPUT_DIR / 's8g_xsec_align.csv'
cmp_df.to_csv(cmp_csv, index=False)
examples_csv = OUTPUT_DIR / 's8g_xsec_ex.csv'
pd.DataFrame(example_rows).to_csv(examples_csv, index=False)

md_lines = ['# Section 8G Cross-Section Alignment Report', '', 'This report compares strict Section VIII challenge evidence with upstream Section V/VI/VII linkage signals.', '']
for _, r in cmp_df.iterrows():
    md_lines.append(f"## {r['challenge_domain']}")
    md_lines.append(f"- strict_evidence_count: {int(r['strict_evidence_count'])}")
    md_lines.append(f"- linked_any_upstream_count: {int(r['linked_any_upstream_count'])}")
    md_lines.append(f"- strict_without_upstream_count: {int(r['strict_without_upstream_count'])}")
    md_lines.append('')

report_md = OUTPUT_DIR / 'section8G_cross_section_report.md'
report_md.write_text('\n'.join(md_lines), encoding='utf-8')

print('Saved:', cmp_csv)
print('Saved:', examples_csv)
print('Saved:', report_md)
print(cmp_df.to_string(index=False))


Saved: analysis/VIII_ev_v1/s8g_xsec_align.csv
Saved: analysis/VIII_ev_v1/s8g_xsec_ex.csv
Saved: analysis/VIII_ev_v1/section8G_cross_section_report.md
                challenge_domain  strict_evidence_count  linked_section5_count  linked_section6_count  linked_section7_count  linked_any_upstream_count  strict_without_upstream_count
standardization_interoperability                     55                     55                     55                     55                         55                              0
 hardware_scalability_efficiency                     25                     25                     25                     25                         25                              0
     channel_modeling_evaluation                     54                     54                     54                     54                         54                              0
    security_privacy_reliability                     18                     18                     18                 

In [16]:
# @title 16. Readiness Report
report_files = [
    'section8A_evidence.csv',
    'section8B_evidence.csv',
    'section8C_evidence.csv',
    'section8D_evidence.csv',
    'section8E_evidence.csv',
    'section8F_research_agenda.csv',
    'section8F_summary_table.csv',
    's8f_dep_cov.csv',
    's8f_pap_chal_map.csv',
    'section8F_summary.json',
    'evidence_graph.jsonl',
    'retrieval_hits.jsonl',
    'anchor_table.csv',
    'axis_definitions.md',
    'mapping_rules.md',
    'cluster_map.csv',
    'contract_violations.csv',
    's8g_xsec_align.csv',
    's8g_xsec_ex.csv',
    'section8G_cross_section_report.md',
]

report = []
for fname in report_files:
    p = OUTPUT_DIR / fname
    report.append(f"{fname}: " + ('OK' if p.exists() else 'MISSING'))

stats = []
try:
    for sec in ['A','B','C','D','E']:
        p = OUTPUT_DIR / f'section8{sec}_evidence.csv'
        if p.exists():
            d = pd.read_csv(p)
            stats.append(f'section8{sec}_rows: {len(d)}')
            stats.append(f'section8{sec}_unique_papers: {d["paper_id"].nunique() if "paper_id" in d.columns else 0}')

    ps = OUTPUT_DIR / 'section8F_summary.json'
    if ps.exists():
        s = json.loads(ps.read_text(encoding='utf-8'))
        for k in ['n_total_papers','n_standardization_interoperability_papers','n_hardware_scalability_efficiency_papers','n_channel_modeling_evaluation_papers','n_security_privacy_reliability_papers','n_deployment_convergence_roadmap_papers','n_multi_challenge_papers','n_all_challenge_domains']:
            stats.append(f'{k}: {s.get(k)}')

    pvc = OUTPUT_DIR / 'contract_violations.csv'
    if pvc.exists():
        v = pd.read_csv(pvc)
        stats.append(f'contract_violations_rows: {len(v)}')
except Exception as e:
    stats.append(f'stats_error: {e}')

report_path = OUTPUT_DIR / 'readiness_report.md'
report_path.write_text('\n'.join(report + [''] + stats), encoding='utf-8')
print('\n'.join(report + [''] + stats))
print('Saved:', report_path)


section8A_evidence.csv: OK
section8B_evidence.csv: OK
section8C_evidence.csv: OK
section8D_evidence.csv: OK
section8E_evidence.csv: OK
section8F_research_agenda.csv: OK
section8F_summary_table.csv: OK
s8f_dep_cov.csv: OK
s8f_pap_chal_map.csv: OK
section8F_summary.json: OK
evidence_graph.jsonl: OK
retrieval_hits.jsonl: OK
anchor_table.csv: OK
axis_definitions.md: OK
mapping_rules.md: OK
cluster_map.csv: OK
contract_violations.csv: OK
s8g_xsec_align.csv: OK
s8g_xsec_ex.csv: OK
section8G_cross_section_report.md: OK

section8A_rows: 625
section8A_unique_papers: 221
section8B_rows: 300
section8B_unique_papers: 221
section8C_rows: 410
section8C_unique_papers: 221
section8D_rows: 276
section8D_unique_papers: 221
section8E_rows: 226
section8E_unique_papers: 221
n_total_papers: 221
n_standardization_interoperability_papers: 55
n_hardware_scalability_efficiency_papers: 25
n_channel_modeling_evaluation_papers: 54
n_security_privacy_reliability_papers: 18
n_deployment_convergence_roadmap_papers: 0